# QRAMP 0.1.1 review demo

Run this notebook from the extracted revised software package. In Colab the first cell asks you to upload that ZIP if the source is not already present. This avoids silently cloning a different version. Install includes all source grammars and CP-SAT. No live endpoint is contacted.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile

def find_project(base):
    candidates = [base, base / "QRAMP"] + list(base.glob("QRAMP*/"))
    return next((p.resolve() for p in candidates if (p / "pyproject.toml").is_file()), None)

project = find_project(Path.cwd()) or find_project(Path("/content"))
if project is None:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        if zipfile.is_zipfile(filename):
            destination = Path("/content").resolve()
            with zipfile.ZipFile(filename) as z:
                for entry in z.namelist():
                    if not (destination / entry).resolve().is_relative_to(destination):
                        raise ValueError("ZIP entry outside extraction directory")
                z.extractall(destination)
    project = find_project(Path("/content"))
if project is None:
    raise RuntimeError("Extract the revised QRAMP ZIP and run from its project directory")
os.chdir(project)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{project}[source,solver,validate,versions,dev]"], check=True)
import qramp
print("Project:", project, "Version:", qramp.__version__)
assert qramp.__version__ == "0.1.1"


## Reproduce inventory, discovery and scoped acceptance

This includes manifests and the 36-month list-scheduler baseline used by the revised experiment script. The controlled challenge set is newly supplied author-constructed evidence, not independent real-project annotation.


In [ ]:
out = project / "revision-results" / "notebook"
subprocess.run([sys.executable, "scripts/reproduce_revision.py", "--out", str(out)], check=True)
import json
summary = json.loads((out / "summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: summary[k] for k in ["scan_assets", "loaded_assets", "occurrences", "tasks", "verification_status_counts"]}, indent=2))


## CP-SAT comparison

One worker, seed 42 and a ten-second wall-clock budget. A feasible result is not a proven optimum; retain the reported bound.


In [ ]:
subprocess.run([sys.executable, "-m", "qramp.cli", "plan", str(out / "assessment.json"), "--capacity", "testbed/policy/teams.yaml", "--horizon", "36m", "--solver", "cpsat", "--time-limit", "10", "-o", str(out / "plan-cpsat.yaml")], check=True)
import yaml
cp = yaml.safe_load((out / "plan-cpsat.yaml").read_text(encoding="utf-8"))
print(cp["solver"], cp["solver_meta"])


## View the report

Readiness concerns available replacement capability, not completed migration. Passing a target check does not assess newly introduced uses elsewhere.


In [ ]:
from IPython.display import HTML, display
import html
dashboard = (out / "dashboard.html").read_text(encoding="utf-8")
display(HTML('<iframe width="100%" height="700" srcdoc="' + html.escape(dashboard, quote=True) + '"></iframe>'))


## Regression tests


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "-rs"], check=True)


## Optional full scheduling sensitivity

Enable the flag to run three ten-second CP-SAT repetitions for each of seven configurations. All raw schedules and independent constraint checks are saved.


In [ ]:
RUN_SENSITIVITY = False
if RUN_SENSITIVITY:
    subprocess.run([sys.executable, "scripts/reproduce_revision.py", "--out", str(out), "--sensitivity", "--repeats", "3", "--time-limit", "10"], check=True)
